# Diagnostic v3 - No Packing

Test with max_seq_length=512 but packing=False

In [ ]:
%%capture
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
import os, time, json, sys
os.chdir("/kaggle/working")
if not os.path.exists("fingpt-qlora"):
    !git clone https://github.com/W-Kaski/fingpt-qlora.git
os.chdir("fingpt-qlora")
log_path = "results/diagnostic_v3_log.txt"
os.makedirs("results", exist_ok=True)
class Tee:
    def __init__(self, *files): self.files = files
    def write(self, obj):
        for f in self.files: f.write(obj); f.flush()
    def flush(self):
        for f in self.files: f.flush()
log_file = open(log_path, "w")
sys.stdout = Tee(sys.__stdout__, log_file)
sys.stderr = Tee(sys.__stderr__, log_file)
def log(msg): print(f"[{time.strftime("%H:%M:%S")}] {msg}")
log("Diagnostic v3 started")

In [ ]:
log("=== Data ===")
if not os.path.exists("data/splits/train.json"):
    !python -m src.data.download
    !python -m src.data.preprocess
    !python -m src.data.format_chat
    !python -m src.data.merge_datasets
    !python -m src.data.splits
with open("data/splits/train.json") as f: train_data = json.load(f)
with open("data/splits/val.json") as f: val_data = json.load(f)
log(f"Train: {len(train_data)}, Val: {len(val_data)}")

In [ ]:
import torch
log(f"GPU: {torch.cuda.get_device_name(0)}")
from unsloth import FastLanguageModel
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 512
model, tokenizer = FastLanguageModel.from_pretrained(model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=16, target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"], lora_alpha=32, lora_dropout=0, bias="none", use_gradient_checkpointing="unsloth", random_state=42)

In [ ]:
from datasets import Dataset
def load_split(path):
    with open(path) as f: data = json.load(f)
    texts = []
    for ex in data:
        msgs = []
        for t in ex["conversations"]:
            role = "assistant" if t["from"] == "gpt" else t["from"]
            msgs.append({"role": role, "content": t["value"]})
        texts.append({"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)})
    return Dataset.from_list(texts)
train_dataset = load_split("data/splits/train.json")
val_dataset = load_split("data/splits/val.json")
log(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")
lengths = [tokenizer(train_dataset[i]["text"], return_tensors="pt")["input_ids"].shape[1] for i in range(min(50, len(train_dataset)))]
log(f"Token lengths: min={min(lengths)}, max={max(lengths)}, avg={sum(lengths)//len(lengths)}")

In [ ]:
log("=== Training Test (10 steps, NO packing) ===")
from trl import SFTTrainer, SFTConfig
training_args = SFTConfig(
    output_dir="outputs/diagnostic_v3",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    max_steps=10,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    max_seq_length=MAX_SEQ_LENGTH,
    fp16=True,
    logging_steps=1,
    save_strategy="no",
    eval_strategy="no",
    seed=42,
    report_to="none",
    dataset_text_field="text",
    packing=False,  # NO packing
)
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_dataset, args=training_args)
log("Starting 10 steps...")
t0 = time.time()
trainer.train()
t1 = time.time()
total = t1 - t0
per_step = total / 10
log(f"10 steps: {total:.1f}s")
log(f"Time/step: {per_step:.2f}s")
log(f"Baseline (2048): 9.72s/step")
log(f"Speedup: {9.72 / per_step:.1f}x")
for e in trainer.state.log_history:
    if "loss" in e: log(f"Step {e["step"]}: loss={e["loss"]:.4f}")

In [ ]:
log("=" * 60)
log("SUMMARY v3")
log("=" * 60)
log(f"Config: max_seq_length=512, packing=False")
log(f"Time/step: {per_step:.2f}s")
log(f"Baseline: 9.72s/step")
log(f"Speedup: {9.72 / per_step:.1f}x")
if per_step < 3: log("✅ PASS")
elif per_step < 5: log("⚠️  ACCEPTABLE")
else: log("❌ FAIL")
log(f"Log: {log_path}")